In [ ]:
"""
=============================================================================
ACTUALIZADOR DE df_fear_greed_MERGED - DATOS DE CRIPTOMONEDAS
Data provided by CoinGecko (https://www.coingecko.com)
Fear & Greed from Alternative.me
=============================================================================
"""

import os
import requests
import pandas as pd
import time
from datetime import datetime

# Configuración
BASE_DIR = r"C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF"
RUTA_CSV = os.path.join(BASE_DIR, "data", "csv", "df_fear_greed_merged.csv")

# APIs
COINGECKO_BASE = "https://api.coingecko.com/api/v3"

print("✓ Configuración cargada")
print(f"  CSV a actualizar: {RUTA_CSV}")

✓ Configuración cargada
  CSV a actualizar: C:\Users\josit\CUARTO CURSO\TFG\TFG_Crypto_DEF\data\csv\df_merged.csv


In [3]:
# ==========================================================
# 📊 Obtener datos del Fear & Greed Index de Alternative.me
# ==========================================================
url = "https://api.alternative.me/fng/"
params = {"limit": 5000, "format": "json"}

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()["data"]
    df_fear_greed = pd.DataFrame(data)

    # Convertir tipos y limpiar
    df_fear_greed["timestamp"] = pd.to_datetime(df_fear_greed["timestamp"].astype(int), unit="s")
    df_fear_greed["value"] = pd.to_numeric(df_fear_greed["value"])
    df_fear_greed = df_fear_greed.rename(columns={
        "timestamp": "Date",
        "value": "FearGreed_Value",
        "value_classification": "FearGreed_Label"
    })
    df_fear_greed = df_fear_greed.sort_values("Date")

    # Guardar
    # Cambia esta ruta por la que prefieras:
    ruta_guardado = "../data/csv/raw/fear_greed_index.csv"

    # Crear carpeta si no existe
    os.makedirs(os.path.dirname(ruta_guardado), exist_ok=True)

    # Guardar el archivo CSV
    df_fear_greed.to_csv(ruta_guardado)
    print("✅ Fear & Greed Index descargado correctamente")
    print(df_fear_greed.head())
else:
    print("❌ Error al obtener datos:", response.status_code, response.text)


✅ Fear & Greed Index descargado correctamente
      FearGreed_Value FearGreed_Label       Date time_until_update
3007               30            Fear 2018-02-01               NaN
3006               15    Extreme Fear 2018-02-02               NaN
3005               40            Fear 2018-02-03               NaN
3004               24    Extreme Fear 2018-02-04               NaN
3003               11    Extreme Fear 2018-02-05               NaN


In [5]:
# Cargar el CSV existente
df_merged = pd.read_csv(RUTA_CSV, parse_dates=["date"], index_col="date")
df_merged.sort_index(inplace=True)

print(f"✓ CSV cargado: {len(df_merged)} registros")
print(f"  Última fecha: {df_merged.index.max().date()}")
print(f"  Rango: {df_merged.index.min().date()} → {df_merged.index.max().date()}")

# Mostrar últimas filas
df_merged.tail()

✓ CSV cargado: 2885 registros
  Última fecha: 2025-12-25
  Rango: 2018-02-01 → 2025-12-25


,btc_open,btc_high,btc_low,btc_close,btc_volume,eth_open,eth_high,eth_low,eth_close,eth_volume,btc_dominance,eth_dominance,alt_dominance,fear_greed,FearGreed_Label,inflation,btc_mcap,eth_mcap,fed_rate
date,,,,,,,,,,,,,,,,,,,
2025-12-21,88344.703125,89027.953125,87613.203125,88621.750000,19845522660,2977.508545,3009.807129,2945.394775,3001.613770,11178096676,0.572387,0.116629,0.310984,20.0,Extreme Fear,2.7,1.762805e+12,3.591869e+11,3.84
2025-12-22,88621.398438,90501.929688,87908.070312,88490.015625,38047472118,3001.612793,3073.353760,2965.175049,3006.073730,20342034264,0.573053,0.117441,0.309506,25.0,Extreme Fear,2.7,1.768920e+12,3.625196e+11,3.84
2025-12-23,88490.031250,88898.382812,86606.976562,87414.000000,43683011533,3006.073975,3033.196289,2902.333008,2963.374023,21453213168,0.573204,0.117729,0.309067,24.0,Extreme Fear,2.7,1.767081e+12,3.629357e+11,3.84
2025-12-24,87404.320312,87956.882812,86411.796875,87611.960938,25550297986,2962.922607,2975.171143,2888.988037,2945.590576,13984770325,0.573230,0.117515,0.309256,24.0,Extreme Fear,2.7,1.745209e+12,3.577764e+11,3.84
2025-12-25,87608.320312,88501.789062,86949.257812,87234.742188,19953216347,2945.423096,2968.705078,2891.108887,2903.579590,11828871179,0.574597,0.116787,0.308616,23.0,Extreme Fear,2.7,1.749379e+12,3.555614e+11,3.84


In [ ]:
# Calcular cuántos días faltan por actualizar
ultima_fecha = df_merged.index.max()
hoy = pd.Timestamp.now().normalize()
dias_faltantes = (hoy - ultima_fecha).days

print(f"📅 Última fecha en CSV: {ultima_fecha.date()}")
print(f"📅 Hoy: {hoy.date()}")
print(f"📊 Días a descargar: {dias_faltantes}")

if dias_faltantes <= 0:
    print("\n⚠️ El CSV ya está actualizado")
else:
    print(f"\n✓ Proceder a descargar {dias_faltantes} días de datos")

📅 Última fecha en CSV: 2025-12-25
📅 Hoy: 2026-05-01
📊 Días a descargar: 127

✓ Proceder a descargar 127 días de datos


In [13]:
"""
Data provided by CoinGecko
"""

print("📥 Descargando datos de BITCOIN...")

# ============================================================================
# FUNCIÓN PARA AJUSTAR DÍAS
# ============================================================================

def ajustar_dias_coingecko(dias_necesarios):
    """CoinGecko solo acepta: 1, 7, 14, 30, 90, 180, 365"""
    valores_validos = [1, 7, 14, 30, 90, 180, 365]
    for valor in valores_validos:
        if dias_necesarios <= valor:
            return valor
    return 365

dias_validos = ajustar_dias_coingecko(dias_faltantes)
print(f"  Días a solicitar (ajustado): {dias_validos}")

# ============================================================================
# Market Cap + Volume + PRECIO de Bitcoin (endpoint más completo)
# ============================================================================

url_btc_market = f"{COINGECKO_BASE}/coins/bitcoin/market_chart"
response = requests.get(
    url_btc_market,
    params={"vs_currency": "usd", "days": dias_validos, "interval": "daily"}
)

if response.status_code == 200:
    btc_market = response.json()
    
    # Precio (lo usaremos como "close")
    btc_prices = pd.DataFrame(btc_market["prices"], columns=["timestamp", "price"])
    btc_prices["date"] = pd.to_datetime(btc_prices["timestamp"], unit="ms")
    btc_prices = btc_prices.drop("timestamp", axis=1).set_index("date")
    
    # Volume
    btc_volumes = pd.DataFrame(btc_market["total_volumes"], columns=["timestamp", "volume"])
    btc_volumes["date"] = pd.to_datetime(btc_volumes["timestamp"], unit="ms")
    btc_volumes = btc_volumes.drop("timestamp", axis=1).set_index("date")
    
    # Market Cap
    btc_mcaps = pd.DataFrame(btc_market["market_caps"], columns=["timestamp", "market_cap"])
    btc_mcaps["date"] = pd.to_datetime(btc_mcaps["timestamp"], unit="ms")
    btc_mcaps = btc_mcaps.drop("timestamp", axis=1).set_index("date")
    
    print(f"  ✓ Price: {len(btc_prices)} días")
    print(f"  ✓ Volume: {len(btc_volumes)} días")
    print(f"  ✓ Market Cap: {len(btc_mcaps)} días")
    
    # Combinar todo
    df_btc = btc_prices.join(btc_volumes).join(btc_mcaps)
    
else:
    print(f"  ❌ Error Market: {response.status_code}")
    df_btc = pd.DataFrame()

time.sleep(2)  # Rate limiting

# ============================================================================
# INTENTAR descargar OHLC (open, high, low) - opcional
# ============================================================================

ohlc_disponible = False

url_btc_ohlc = f"{COINGECKO_BASE}/coins/bitcoin/ohlc"
response = requests.get(url_btc_ohlc, params={"vs_currency": "usd", "days": dias_validos})

if response.status_code == 200 and not df_btc.empty:
    try:
        btc_ohlc = response.json()
        df_btc_ohlc = pd.DataFrame(btc_ohlc, columns=["timestamp", "open", "high", "low", "close_ohlc"])
        df_btc_ohlc["date"] = pd.to_datetime(df_btc_ohlc["timestamp"], unit="ms")
        df_btc_ohlc = df_btc_ohlc.drop(["timestamp", "close_ohlc"], axis=1).set_index("date")
        
        # Hacer LEFT JOIN (priorizar datos de market_chart)
        df_btc = df_btc.join(df_btc_ohlc, how="left")
        
        ohlc_disponible = True
        print(f"  ✓ OHLC añadido")
    except Exception as e:
        print(f"  ⚠️ Error parseando OHLC: {e}")
        ohlc_disponible = False
else:
    print(f"  ⚠️ OHLC no disponible (error {response.status_code})")

# ============================================================================
# Completar OHLC faltante usando el precio
# ============================================================================

if not df_btc.empty:
    if ohlc_disponible and "open" in df_btc.columns:
        # Hay algunos datos OHLC, completar los NaN con price
        print(f"  🔧 Completando OHLC con price donde falte...")
        df_btc["close"] = df_btc["price"]  # Usar siempre price como close
        df_btc["open"] = df_btc["open"].fillna(df_btc["price"])
        df_btc["high"] = df_btc["high"].fillna(df_btc["price"])
        df_btc["low"] = df_btc["low"].fillna(df_btc["price"])
    else:
        # No hay OHLC, usar price como aproximación
        print(f"  🔧 Creando OHLC desde precio (OHLC no disponible)...")
        df_btc["close"] = df_btc["price"]
        df_btc["open"] = df_btc["price"]
        df_btc["high"] = df_btc["price"]
        df_btc["low"] = df_btc["price"]
    
    # Eliminar columna price (ya tenemos close)
    df_btc = df_btc.drop("price", axis=1)
    
    # Reordenar columnas: open, high, low, close, volume, market_cap
    df_btc = df_btc[["open", "high", "low", "close", "volume", "market_cap"]]
    
    # Renombrar con prefijo btc_
    df_btc.columns = ["btc_" + col for col in df_btc.columns]
    
    # Verificar NaN
    nan_count = df_btc.isna().sum().sum()
    
    print(f"\n✓ Bitcoin completo: {len(df_btc)} registros")
    print(f"  Total NaN: {nan_count}")

df_btc.tail(10)

📥 Descargando datos de BITCOIN...
  Días a solicitar (ajustado): 180
  ✓ Price: 181 días
  ✓ Volume: 181 días
  ✓ Market Cap: 181 días
  ✓ OHLC añadido
  🔧 Completando OHLC con price donde falte...

✓ Bitcoin completo: 181 registros
  Total NaN: 0


,btc_open,btc_high,btc_low,btc_close,btc_volume,btc_market_cap
date,,,,,,
2026-04-23 00:00:00,78194.783349,78194.783349,78194.783349,78194.783349,4.981527e+10,1.566077e+12
2026-04-24 00:00:00,78260.619980,78260.619980,78260.619980,78260.619980,4.170480e+10,1.566738e+12
2026-04-25 00:00:00,75885.000000,79389.000000,74935.000000,77444.795637,3.358495e+10,1.550504e+12
2026-04-26 00:00:00,77619.142199,77619.142199,77619.142199,77619.142199,1.727668e+10,1.553989e+12
2026-04-27 00:00:00,78645.125602,78645.125602,78645.125602,78645.125602,2.294441e+10,1.575065e+12
2026-04-28 00:00:00,77361.299110,77361.299110,77361.299110,77361.299110,3.945377e+10,1.549703e+12
2026-04-29 00:00:00,77490.000000,79400.000000,75706.000000,76345.225210,3.268317e+10,1.528431e+12
2026-04-30 00:00:00,75774.885523,75774.885523,75774.885523,75774.885523,4.252829e+10,1.517195e+12
2026-05-01 00:00:00,76286.577110,76286.577110,76286.577110,76286.577110,3.171787e+10,1.528164e+12


In [14]:
"""
Data provided by CoinGecko
"""

print("📥 Descargando datos de ETHEREUM...")

time.sleep(2)  # Rate limiting

# ============================================================================
# Market Cap + Volume + PRECIO de Ethereum (endpoint más completo)
# ============================================================================

url_eth_market = f"{COINGECKO_BASE}/coins/ethereum/market_chart"
response = requests.get(
    url_eth_market,
    params={"vs_currency": "usd", "days": dias_validos, "interval": "daily"}
)

if response.status_code == 200:
    eth_market = response.json()
    
    # Precio (lo usaremos como "close")
    eth_prices = pd.DataFrame(eth_market["prices"], columns=["timestamp", "price"])
    eth_prices["date"] = pd.to_datetime(eth_prices["timestamp"], unit="ms")
    eth_prices = eth_prices.drop("timestamp", axis=1).set_index("date")
    
    # Volume
    eth_volumes = pd.DataFrame(eth_market["total_volumes"], columns=["timestamp", "volume"])
    eth_volumes["date"] = pd.to_datetime(eth_volumes["timestamp"], unit="ms")
    eth_volumes = eth_volumes.drop("timestamp", axis=1).set_index("date")
    
    # Market Cap
    eth_mcaps = pd.DataFrame(eth_market["market_caps"], columns=["timestamp", "market_cap"])
    eth_mcaps["date"] = pd.to_datetime(eth_mcaps["timestamp"], unit="ms")
    eth_mcaps = eth_mcaps.drop("timestamp", axis=1).set_index("date")
    
    print(f"  ✓ Price: {len(eth_prices)} días")
    print(f"  ✓ Volume: {len(eth_volumes)} días")
    print(f"  ✓ Market Cap: {len(eth_mcaps)} días")
    
    # Combinar todo
    df_eth = eth_prices.join(eth_volumes).join(eth_mcaps)
    
else:
    print(f"  ❌ Error Market: {response.status_code}")
    df_eth = pd.DataFrame()

time.sleep(2)

# ============================================================================
# INTENTAR descargar OHLC (open, high, low) - opcional
# ============================================================================

ohlc_disponible = False

url_eth_ohlc = f"{COINGECKO_BASE}/coins/ethereum/ohlc"
response = requests.get(url_eth_ohlc, params={"vs_currency": "usd", "days": dias_validos})

if response.status_code == 200 and not df_eth.empty:
    try:
        eth_ohlc = response.json()
        df_eth_ohlc = pd.DataFrame(eth_ohlc, columns=["timestamp", "open", "high", "low", "close_ohlc"])
        df_eth_ohlc["date"] = pd.to_datetime(df_eth_ohlc["timestamp"], unit="ms")
        df_eth_ohlc = df_eth_ohlc.drop(["timestamp", "close_ohlc"], axis=1).set_index("date")
        
        # Hacer LEFT JOIN
        df_eth = df_eth.join(df_eth_ohlc, how="left")
        
        ohlc_disponible = True
        print(f"  ✓ OHLC añadido")
    except Exception as e:
        print(f"  ⚠️ Error parseando OHLC: {e}")
        ohlc_disponible = False
else:
    print(f"  ⚠️ OHLC no disponible (error {response.status_code})")

# ============================================================================
# Completar OHLC faltante usando el precio
# ============================================================================

if not df_eth.empty:
    if ohlc_disponible and "open" in df_eth.columns:
        # Hay algunos datos OHLC, completar los NaN con price
        print(f"  🔧 Completando OHLC con price donde falte...")
        df_eth["close"] = df_eth["price"]  # Usar siempre price como close
        df_eth["open"] = df_eth["open"].fillna(df_eth["price"])
        df_eth["high"] = df_eth["high"].fillna(df_eth["price"])
        df_eth["low"] = df_eth["low"].fillna(df_eth["price"])
    else:
        # No hay OHLC, usar price como aproximación
        print(f"  🔧 Creando OHLC desde precio (OHLC no disponible)...")
        df_eth["close"] = df_eth["price"]
        df_eth["open"] = df_eth["price"]
        df_eth["high"] = df_eth["price"]
        df_eth["low"] = df_eth["price"]
    
    # Eliminar columna price
    df_eth = df_eth.drop("price", axis=1)
    
    # Reordenar columnas
    df_eth = df_eth[["open", "high", "low", "close", "volume", "market_cap"]]
    
    # Renombrar con prefijo eth_
    df_eth.columns = ["eth_" + col for col in df_eth.columns]
    
    # Verificar NaN
    nan_count = df_eth.isna().sum().sum()
    
    print(f"\n✓ Ethereum completo: {len(df_eth)} registros")
    print(f"  Total NaN: {nan_count}")

df_eth.head(10)

📥 Descargando datos de ETHEREUM...
  ✓ Price: 181 días
  ✓ Volume: 181 días
  ✓ Market Cap: 181 días
  ✓ OHLC añadido
  🔧 Completando OHLC con price donde falte...

✓ Ethereum completo: 181 registros
  Total NaN: 0


,eth_open,eth_high,eth_low,eth_close,eth_volume,eth_market_cap
date,,,,,,
2025-11-03,3910.094769,3910.094769,3910.094769,3910.094769,1.551162e+10,4.715494e+11
2025-11-04,3801.550000,3911.960000,3572.500000,3600.715502,4.791406e+10,4.341497e+11
2025-11-05,3296.743760,3296.743760,3296.743760,3296.743760,6.878429e+10,3.974893e+11
2025-11-06,3427.690591,3427.690591,3427.690591,3427.690591,4.437453e+10,4.139187e+11
2025-11-07,3308.918165,3308.918165,3308.918165,3308.918165,3.388097e+10,3.990074e+11
2025-11-08,3600.360000,3649.010000,3097.710000,3434.351229,3.897260e+10,4.146914e+11
2025-11-09,3401.459400,3401.459400,3401.459400,3401.459400,2.087697e+10,4.105991e+11
2025-11-10,3576.254705,3576.254705,3576.254705,3576.254705,2.599421e+10,4.312943e+11
2025-11-11,3564.610596,3564.610596,3564.610596,3564.610596,3.186656e+10,4.302720e+11


In [15]:
"""
Data provided by CoinGecko
"""

print("📥 Descargando DOMINANCIAS globales...")

time.sleep(2)

url_global = f"{COINGECKO_BASE}/global"
response = requests.get(url_global)

if response.status_code == 200:
    global_data = response.json()["data"]
    
    btc_dom = global_data["market_cap_percentage"].get("btc", 0)
    eth_dom = global_data["market_cap_percentage"].get("eth", 0)
    alt_dom = 100 - btc_dom - eth_dom
    
    dominancias = {
        "btc_dominance": btc_dom / 100,
        "eth_dominance": eth_dom / 100,
        "alt_dominance": alt_dom / 100,
    }
    
    print(f"  ✓ BTC Dominance: {btc_dom:.2f}%")
    print(f"  ✓ ETH Dominance: {eth_dom:.2f}%")
    print(f"  ✓ ALT Dominance: {alt_dom:.2f}%")
else:
    print(f"  ❌ Error: {response.status_code}")
    # Usar último valor conocido del CSV como fallback
    dominancias = {
        "btc_dominance": df_merged["btc_dominance"].iloc[-1],
        "eth_dominance": df_merged["eth_dominance"].iloc[-1],
        "alt_dominance": df_merged["alt_dominance"].iloc[-1],
    }
    print(f"  ⚠️ Usando último valor del CSV")

print("\n📊 Dominancias:")
dominancias

📥 Descargando DOMINANCIAS globales...
  ✓ BTC Dominance: 58.46%
  ✓ ETH Dominance: 10.38%
  ✓ ALT Dominance: 31.16%

📊 Dominancias:


{'btc_dominance': 0.5846442264910322,
 'eth_dominance': 0.10380089482380823,
 'alt_dominance': 0.3115548786851596}

In [17]:
print("="*70)
print("RESUMEN DE DATOS DESCARGADOS")
print("="*70)

print(f"\n📊 Bitcoin:")
print(f"   Filas: {len(df_btc)}")
print(f"   Columnas: {list(df_btc.columns)}")
print(f"   Rango: {df_btc.index.min().date()} → {df_btc.index.max().date()}")
print(f"   NaN totales: {df_btc.isna().sum().sum()}")

print(f"\n📊 Ethereum:")
print(f"   Filas: {len(df_eth)}")
print(f"   Columnas: {list(df_eth.columns)}")
print(f"   Rango: {df_eth.index.min().date()} → {df_eth.index.max().date()}")
print(f"   NaN totales: {df_eth.isna().sum().sum()}")


print(f"\n📊 Dominancias:")
for k, v in dominancias.items():
    print(f"   {k}: {v:.4f}")

print("\n" + "="*70)

RESUMEN DE DATOS DESCARGADOS

📊 Bitcoin:
   Filas: 181
   Columnas: ['btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume', 'btc_market_cap']
   Rango: 2025-11-03 → 2026-05-01
   NaN totales: 0

📊 Ethereum:
   Filas: 181
   Columnas: ['eth_open', 'eth_high', 'eth_low', 'eth_close', 'eth_volume', 'eth_market_cap']
   Rango: 2025-11-03 → 2026-05-01
   NaN totales: 0

📊 Dominancias:
   btc_dominance: 0.5846
   eth_dominance: 0.1038
   alt_dominance: 0.3116



In [18]:
"""
Data provided by CoinGecko
"""

print("📥 Descargando DATOS GLOBALES (market cap total + dominancias)...")

time.sleep(2)

url_global = f"{COINGECKO_BASE}/global"
response = requests.get(url_global)

if response.status_code == 200:
    global_data = response.json()["data"]
    
    # Dominancias actuales
    btc_dom = global_data["market_cap_percentage"].get("btc", 0)
    eth_dom = global_data["market_cap_percentage"].get("eth", 0)
    alt_dom = 100 - btc_dom - eth_dom
    
    # MARKET CAP TOTAL (esto es lo que necesitamos)
    total_market_cap = global_data["total_market_cap"]["usd"]
    
    datos_globales = {
        "btc_dominance": btc_dom / 100,
        "eth_dominance": eth_dom / 100,
        "alt_dominance": alt_dom / 100,
        "total_market_cap": total_market_cap,  # ← NUEVO
    }
    
    print(f"  ✓ BTC Dominance: {btc_dom:.2f}%")
    print(f"  ✓ ETH Dominance: {eth_dom:.2f}%")
    print(f"  ✓ ALT Dominance: {alt_dom:.2f}%")
    print(f"  ✓ Total Market Cap: ${total_market_cap:,.0f}")
else:
    print(f"  ❌ Error: {response.status_code}")
    # Usar último valor conocido del CSV como fallback
    datos_globales = {
        "btc_dominance": df_merged["btc_dominance"].iloc[-1],
        "eth_dominance": df_merged["eth_dominance"].iloc[-1],
        "alt_dominance": df_merged["alt_dominance"].iloc[-1],
        "total_market_cap": None,
    }
    print(f"  ⚠️ Usando último valor del CSV")

print("\n📊 Datos globales:")
datos_globales

📥 Descargando DATOS GLOBALES (market cap total + dominancias)...
  ✓ BTC Dominance: 58.46%
  ✓ ETH Dominance: 10.38%
  ✓ ALT Dominance: 31.16%
  ✓ Total Market Cap: $2,680,145,465,793

📊 Datos globales:


{'btc_dominance': 0.5846442264910322,
 'eth_dominance': 0.10380089482380823,
 'alt_dominance': 0.3115548786851596,
 'total_market_cap': 2680145465792.999}

In [21]:
print("🔧 Combinando todos los datos...")

# 1. Combinar BTC + ETH
df_nuevo = df_btc.join(df_eth, how="outer")
print(f"  ✓ BTC + ETH: {len(df_nuevo)} registros")

# 2. CALCULAR DOMINANCIAS DÍA A DÍA usando market cap total real
print(f"\n🔧 Calculando dominancias día a día...")

if datos_globales["total_market_cap"] is not None:
    # MÉTODO 1: Usar ratio actual para estimar total market cap histórico
    # Obtenemos el último día de datos nuevos
    ultimo_dia_nuevo = df_nuevo.iloc[-1]
    btc_mcap_ultimo = ultimo_dia_nuevo["btc_market_cap"]  # ← CORREGIDO
    eth_mcap_ultimo = ultimo_dia_nuevo["eth_market_cap"]  # ← CORREGIDO
    
    # Market cap total actual (de la API)
    total_mcap_actual = datos_globales["total_market_cap"]
    
    # Calcular market cap de altcoins actual
    alt_mcap_actual = total_mcap_actual - btc_mcap_ultimo - eth_mcap_ultimo
    
    print(f"  📊 Valores del día más reciente:")
    print(f"     BTC mcap: ${btc_mcap_ultimo:,.0f}")
    print(f"     ETH mcap: ${eth_mcap_ultimo:,.0f}")
    print(f"     Total mcap (API): ${total_mcap_actual:,.0f}")
    print(f"     ALT mcap (calculado): ${alt_mcap_actual:,.0f}")
    
    # Para cada día, calcular dominancias
    dominancias_list = []
    
    for idx, row in df_nuevo.iterrows():
        btc_mcap = row["btc_market_cap"]  # ← CORREGIDO
        eth_mcap = row["eth_market_cap"]  # ← CORREGIDO
        
        # Estimar alt_mcap para este día
        # Asumimos que alt_mcap cambia proporcionalmente al promedio de BTC y ETH
        cambio_btc = btc_mcap / btc_mcap_ultimo if btc_mcap_ultimo > 0 else 1
        cambio_eth = eth_mcap / eth_mcap_ultimo if eth_mcap_ultimo > 0 else 1
        cambio_promedio = (cambio_btc + cambio_eth) / 2
        
        alt_mcap_estimado = alt_mcap_actual * cambio_promedio
        total_mcap_estimado = btc_mcap + eth_mcap + alt_mcap_estimado
        
        # Calcular dominancias
        btc_dom = btc_mcap / total_mcap_estimado if total_mcap_estimado > 0 else 0
        eth_dom = eth_mcap / total_mcap_estimado if total_mcap_estimado > 0 else 0
        alt_dom = alt_mcap_estimado / total_mcap_estimado if total_mcap_estimado > 0 else 0
        
        dominancias_list.append({
            "btc_dominance": btc_dom,
            "eth_dominance": eth_dom,
            "alt_dominance": alt_dom
        })
    
    # Crear DataFrame con dominancias
    df_dominancias = pd.DataFrame(dominancias_list, index=df_nuevo.index)
    
    # Añadir al DataFrame principal
    df_nuevo["btc_dominance"] = df_dominancias["btc_dominance"]
    df_nuevo["eth_dominance"] = df_dominancias["eth_dominance"]
    df_nuevo["alt_dominance"] = df_dominancias["alt_dominance"]
    
    print(f"  ✓ Dominancias calculadas día a día")
    
else:
    # Fallback: usar valores del CSV
    print(f"  ⚠️ Sin market cap total, usando forward-fill")
    df_nuevo["btc_dominance"] = df_merged["btc_dominance"].iloc[-1]
    df_nuevo["eth_dominance"] = df_merged["eth_dominance"].iloc[-1]
    df_nuevo["alt_dominance"] = df_merged["alt_dominance"].iloc[-1]

print(f"\n  📊 Estadísticas de dominancias calculadas:")
print(f"     BTC - min: {df_nuevo['btc_dominance'].min():.4f} | max: {df_nuevo['btc_dominance'].max():.4f} | último: {df_nuevo['btc_dominance'].iloc[-1]:.4f}")
print(f"     ETH - min: {df_nuevo['eth_dominance'].min():.4f} | max: {df_nuevo['eth_dominance'].max():.4f} | último: {df_nuevo['eth_dominance'].iloc[-1]:.4f}")
print(f"     ALT - min: {df_nuevo['alt_dominance'].min():.4f} | max: {df_nuevo['alt_dominance'].max():.4f} | último: {df_nuevo['alt_dominance'].iloc[-1]:.4f}")

# 3. Añadir Fear & Greed (merge por fecha)
if not df_feargreed.empty:
    df_nuevo = df_nuevo.join(df_feargreed, how="left")
    
    # Forward-fill para fechas sin datos de Fear & Greed
    df_nuevo["fear_greed"] = df_nuevo["fear_greed"].ffill()
    df_nuevo["FearGreed_Label"] = df_nuevo["FearGreed_Label"].ffill()
    
    print(f"  ✓ Fear & Greed añadido")
else:
    # Si no hay Fear & Greed, forward-fill del CSV
    df_nuevo["fear_greed"] = df_merged["fear_greed"].iloc[-1]
    df_nuevo["FearGreed_Label"] = df_merged["FearGreed_Label"].iloc[-1]
    print(f"  ⚠️ Fear & Greed usando último valor del CSV")

# 4. Añadir inflation y fed_rate (forward-fill del último valor del CSV)
df_nuevo["inflation"] = df_merged["inflation"].iloc[-1]
df_nuevo["fed_rate"] = df_merged["fed_rate"].iloc[-1]
print(f"  ✓ Inflation y Fed Rate (forward-filled)")

# 5. Renombrar market_cap a mcap para match con tu CSV
df_nuevo = df_nuevo.rename(columns={
    "btc_market_cap": "btc_mcap",
    "eth_market_cap": "eth_mcap"
})

# 6. Reordenar columnas para match exacto con df_merged
columnas_ordenadas = [
    "btc_open", "btc_high", "btc_low", "btc_close", "btc_volume",
    "eth_open", "eth_high", "eth_low", "eth_close", "eth_volume",
    "btc_dominance", "eth_dominance", "alt_dominance",
    "fear_greed", "FearGreed_Label",
    "inflation",
    "btc_mcap", "eth_mcap",
    "fed_rate"
]

df_nuevo = df_nuevo[columnas_ordenadas]

print(f"\n✓ DataFrame nuevo formateado: {len(df_nuevo)} registros")
print(f"  Columnas: {len(df_nuevo.columns)} (match con df_merged)")
print(f"  NaN totales: {df_nuevo.isna().sum().sum()}")

# Mostrar evolución de dominancias en los primeros días
print(f"\n📊 Primeros 5 días (verificación):")
df_nuevo.head()

🔧 Combinando todos los datos...
  ✓ BTC + ETH: 182 registros

🔧 Calculando dominancias día a día...
  📊 Valores del día más reciente:
     BTC mcap: $nan
     ETH mcap: $278,207,132,796
     Total mcap (API): $2,680,145,465,793
     ALT mcap (calculado): $nan
  ✓ Dominancias calculadas día a día

  📊 Estadísticas de dominancias calculadas:
     BTC - min: 0.0000 | max: 0.0000 | último: 0.0000
     ETH - min: 0.0000 | max: 0.0000 | último: 0.0000
     ALT - min: 0.0000 | max: 0.0000 | último: 0.0000


NameError: name 'df_feargreed' is not defined